# Modelado Avanzado: Stretched Exponential y ML Híbrido

Como vimos, el modelo Arps choca contra límites físicos ($b=2.5$) y sacrifica el ajuste de la cola (mes 20+) para poder capturar el pico. En este notebook solucionamos esa deficiencia fundamental de Vaca Muerta usando dos técnicas avanzadas:

1. **Física Alternativa (Stretched Exponential):** Un modelo con un grado de libertad extra que permite una caída inicial brutal sin perder el ajuste a largo plazo.
2. **Data Science (Machine Learning Híbrido):** Usamos la curva física como línea base (macro-tendencia) y entrenamos un `RandomForestRegressor` sobre los **residuos** para aprender el ruido operativo (shut-ins, rebotes) que la física no puede predecir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.models.arps import ArpsModeler

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 1. Cargar datos del pozo conflictivo
df = pd.read_parquet(project_root / 'data' / 'processed' / 'vaca_muerta_prod.parquet')
df_ypf = df[df['empresa'].str.contains('YPF', na=False, case=False)]
sigla_pozo = 'YPF.Nq.SOil-435(h)'

df_pozo = df_ypf[df_ypf['sigla'] == sigla_pozo].sort_values('mes_relativo')
meses = df_pozo['mes_relativo'].values
produccion = df_pozo['prod_pet'].values

idx_pico = np.argmax(produccion)
t_raw = meses[idx_pico:]
q_raw = produccion[idx_pico:]
mask = q_raw > 0
t_norm = t_raw[mask] - t_raw[mask][0]
q_norm = q_raw[mask]

In [ ]:
# 2. Ajuste Físico Avanzado: Stretched Exponential
model_stretched = ArpsModeler(model_type='stretched')
model_stretched.fit(t_norm, q_norm)

# Predecir la curva física
q_fisica_pred = model_stretched.predict(t_norm)

# 3. Machine Learning: Modelado de Residuos
residuos = q_norm - q_fisica_pred

# Entrenamos el Random Forest para predecir el residuo en base al tiempo 
# (En la vida real acá agregaríamos presión, tamaño de choke, etc.)
X = t_norm.reshape(-1, 1)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X, residuos)

residuos_pred = rf.predict(X)

# Híbrido = Macro-Tendencia (Física) + Micro-Variación (ML)
q_hibrido = q_fisica_pred + residuos_pred

In [ ]:
# 4. Visualización Comparativa
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(t_norm, q_norm, color='black', alpha=0.6, label='Dato Crudo Real')

ax.plot(t_norm, q_fisica_pred, color='red', linewidth=2.5, linestyle='--', 
        label=f'Física Pura (Stretched Exp: n={model_stretched.params["n"]:.2f})')

ax.plot(t_norm, q_hibrido, color='green', linewidth=2.5, 
        label='Modelo Híbrido (Física + Random Forest)')

ax.set_title(f"Resolución del Underfitting: Stretched Exp y ML Híbrido - {sigla_pozo}", fontsize=14, fontweight='bold')
ax.set_xlabel("Meses de Producción", fontsize=12)
ax.set_ylabel("Caudal de Petróleo (m³/mes)", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()